# UPDATED PCA with saving

In [1]:
import mne
from mne.viz import Brain
import os
import numpy as np
import pickle
import pyarrow

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config import check_paths
from config.config import (HEMI, ROI, FS_SUB, FS_SRC_PATH, COUPLINGS,
                           SUBJECTS, GROUPS, TASKS, TASK_STAGES, TASK_BLOCKS, ADAPTATION_SPLITS, PCA_TMAX, PCA_TMIN)
from config.paths import FS_FOLDER, SOURCE_DATA_DIR, ERPAC_DIR, ROI_STCS_DIR, ERPAC_FIGS_DIR, PCA_ROOT, COMPONENTS_DIR, STCS_ALL_VTX_DIR, STCS_AVE_VTX_DIR
from utils.helpers import iterate_dataset
from utils.plotting import plot_group_erpac, plot_group_erpac_timecourse

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorpac import EventRelatedPac

from sklearn.decomposition import PCA

import hashlib

for directory in (
    COMPONENTS_DIR,
    STCS_ALL_VTX_DIR,
    STCS_AVE_VTX_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============================================================
# PCA DENOISING + ROI TIME-COURSE EXTRACTION
# ============================================================
#
# Pipeline:
#
# ROI source epochs
# (epochs × vertices × times)
#          ↓
# source-polarity sign correction
#          ↓
# fit subject-specific spatial PCA on pooled 0–0.5 s data
#          ↓
# retain minimum number of PCs explaining >= 90% variance
#          ↓
# project each FULL condition through the same PCA model
#          ↓
# reconstruct denoised vertices
# (epochs × original_vertices × times)
#          ↓
# mean across sign-aligned vertices
#          ↓
# (epochs × times)
#          ↓
# add singleton vertex dimension
#          ↓
# (epochs × 1 × times)
#
# Outputs:
#   PCA_ROOT/components/<sub>/<roi>/
#   PCA_ROOT/stcs_all_vtx/<group>/<sub>/<task>/<stage>/[block]/[adapt_split]/
#   PCA_ROOT/stcs_ave_vtx/<group>/<sub>/
#
# ============================================================


# ============================================================
# SETTINGS
# ============================================================

PCA_TMIN = 0.0
PCA_TMAX = 0.5

# Primary denoising criterion.
# Use 0.95 later as a prespecified sensitivity analysis if desired.
PCA_VARIANCE_THRESHOLD = 0.90

# Existing FTT balancing procedure.
FTT_KEEP_FRACTION = 2 / 3
FTT_RANDOM_SEED = 42

# ============================================================
# LOAD FSAVERAGE SOURCE SPACE + LABELS
# ============================================================

print("Loading fsaverage source space...")
src = mne.read_source_spaces(FS_SRC_PATH)
print("Source space loaded.")

print("Loading cortical labels...")

all_labels = mne.read_labels_from_annot(
    subject=FS_SUB,
    parc="aparc.a2009s",
    hemi=HEMI[1:],
    subjects_dir=FS_FOLDER,
)

label_dict = {label.name: label for label in all_labels}

print(f"Loaded {len(all_labels)} anatomical labels.")


# ============================================================
# HELPERS
# ============================================================

def get_combined_roi_label(roi_name):
    """Combine all anatomical labels belonging to one functional ROI."""

    roi_label_names = ROI[roi_name]

    if isinstance(roi_label_names, str):
        roi_label_names = [roi_label_names]

    missing = [
        name
        for name in roi_label_names
        if name not in label_dict
    ]

    if missing:
        raise ValueError(
            f"{roi_name}: labels not found in aparc.a2009s:\n{missing}"
        )

    labels = [label_dict[name] for name in roi_label_names]

    combined = labels[0]
    for label in labels[1:]:
        combined = combined + label

    return combined


def find_stc_files(
    subject,
    task,
    task_stage,
    task_block=None,
    adaptation_split=None,
):
    """
    Find -lh.stc files by matching condition tokens against the FULL PATH.

    Example supported hierarchy:
    .../<group>/<sub>/DeCRAT/plan/adaptation/early/<epoch>-lh.stc
    """

    source_dir = Path(SOURCE_DATA_DIR)

    required_tokens = [
        str(subject).lower(),
        str(task).lower(),
        str(task_stage).lower(),
    ]

    if task_block is not None:
        required_tokens.append(str(task_block).lower())

    if adaptation_split is not None:
        required_tokens.append(str(adaptation_split).lower())

    files = []

    for file in source_dir.rglob("*-lh.stc"):
        full_path = str(file).lower()

        if all(token in full_path for token in required_tokens):
            files.append(file)

    return sorted(files)


def load_condition_stcs(
    subject,
    task,
    task_stage,
    task_block=None,
    adaptation_split=None,
):
    """Load all source epochs for one condition."""

    lh_files = find_stc_files(
        subject=subject,
        task=task,
        task_stage=task_stage,
        task_block=task_block,
        adaptation_split=adaptation_split,
    )

    stcs = []

    for lh_file in lh_files:
        prefix = str(lh_file)[:-7]  # strip "-lh.stc"

        stc = mne.read_source_estimate(
            prefix,
            subject=FS_SUB,
        )

        stcs.append(stc)

    return stcs


def get_conditions():
    """Generate FTT and DeCRAT conditions entering the subject-specific PCA."""

    conditions = []

    for task in TASKS:

        for stage in TASK_STAGES:

            if task == "DeCRAT":

                for block in TASK_BLOCKS:

                    if block == "adaptation":

                        for adaptation_split in ADAPTATION_SPLITS:
                            conditions.append(
                                {
                                    "task": task,
                                    "task_stage": stage,
                                    "task_block": block,
                                    "adaptation_split": adaptation_split,
                                }
                            )

                    else:
                        conditions.append(
                            {
                                "task": task,
                                "task_stage": stage,
                                "task_block": block,
                                "adaptation_split": None,
                            }
                        )

            else:
                conditions.append(
                    {
                        "task": task,
                        "task_stage": stage,
                        "task_block": None,
                        "adaptation_split": None,
                    }
                )

    return conditions


CONDITIONS = get_conditions()


def make_condition_name(task, stage, block=None, adaptation_split=None):
    """Create a stable condition label used in metadata and filenames."""

    parts = [task, stage]

    if block is not None:
        parts.append(block)

    if adaptation_split is not None:
        parts.append(adaptation_split)

    return "_".join(parts)


def subsample_ftt_stcs(
    stcs,
    subject,
    task_stage,
    keep_fraction=FTT_KEEP_FRACTION,
    random_seed=FTT_RANDOM_SEED,
):
    """
    Reproducibly retain a fixed fraction of FTT epochs.

    The seed is subject- and stage-specific, so the same FTT trials
    are retained for every ROI.
    """

    n_total = len(stcs)

    if n_total == 0:
        return stcs, np.array([], dtype=int)

    if not (0 < keep_fraction <= 1):
        raise ValueError("keep_fraction must be > 0 and <= 1.")

    n_keep = max(1, int(np.floor(n_total * keep_fraction)))

    seed_key = f"{random_seed}|{subject}|FTT|{task_stage}"

    stable_seed = int(
        hashlib.sha256(seed_key.encode("utf-8")).hexdigest()[:8],
        16,
    )

    rng = np.random.default_rng(stable_seed)

    selected_indices = np.sort(
        rng.choice(
            n_total,
            size=n_keep,
            replace=False,
        )
    )

    sampled_stcs = [stcs[i] for i in selected_indices]

    return sampled_stcs, selected_indices


def extract_roi_epoch_data(stcs, roi_label, src):
    """
    Extract sign-aligned vertex-level ROI source data.

    Returns
    -------
    data : ndarray
        (n_epochs, n_vertices, n_times)

    vertices_lh, vertices_rh : ndarray
        ROI vertex IDs in each hemisphere.

    flip : ndarray
        MNE source-orientation sign correction (+1 / -1)
        in the same order as the concatenated ROI vertices.
    """

    if len(stcs) == 0:
        raise ValueError("No STCs supplied.")

    first_roi_stc = stcs[0].in_label(roi_label)

    vertices_lh = first_roi_stc.vertices[0].copy()
    vertices_rh = first_roi_stc.vertices[1].copy()

    vertices = np.concatenate([vertices_lh, vertices_rh])

    if len(vertices) == 0:
        raise ValueError(
            f"No vertices found for ROI label {roi_label.name}"
        )

    flip = mne.label_sign_flip(
        roi_label,
        src,
    )

    if len(flip) != len(vertices):
        raise RuntimeError(
            "Mismatch between ROI vertex count and label_sign_flip output.\n"
            f"ROI vertices: {len(vertices)}\n"
            f"flip length: {len(flip)}"
        )

    epoch_data = []

    for epoch_idx, stc in enumerate(stcs):

        roi_stc = stc.in_label(roi_label)

        current_vertices = np.concatenate(
            [
                roi_stc.vertices[0],
                roi_stc.vertices[1],
            ]
        )

        if not np.array_equal(current_vertices, vertices):
            raise RuntimeError(
                f"Vertex mismatch at epoch {epoch_idx}"
            )

        data = roi_stc.data.copy()

        # Sign-align before PCA and before later ROI averaging.
        data *= flip[:, np.newaxis]

        epoch_data.append(data)

    epoch_data = np.stack(epoch_data, axis=0)

    return (
        epoch_data,
        vertices_lh,
        vertices_rh,
        flip,
    )


def components_for_variance(explained_variance_ratio, threshold):
    """Minimum number of PCs required to reach a variance threshold."""

    cumulative = np.cumsum(explained_variance_ratio)

    return (
        np.searchsorted(cumulative, threshold)
        + 1
    )


def get_pca_window_mask(times):
    """Return a robust 0–0.5 s mask for PCA fitting."""

    tol = 1e-9

    mask = (
        (times >= PCA_TMIN - tol)
        &
        (times <= PCA_TMAX + tol)
    )

    if not np.any(mask):
        raise RuntimeError(
            f"No samples found in PCA fitting window "
            f"{PCA_TMIN:.3f}–{PCA_TMAX:.3f} s."
        )

    return mask


def reconstruct_condition(
    roi_data_full,
    pca,
    n_keep,
    good_vertex_mask,
):
    """
    Reconstruct one FULL condition using only the retained PCA subspace.

    PCA was fitted using good vertices only.

    Parameters
    ----------
    roi_data_full : ndarray
        Sign-aligned full epochs:
        (epochs, original_vertices, times)

    pca : sklearn PCA
        Subject × ROI PCA model fitted on pooled 0–0.5 s data.

    n_keep : int
        Number of leading PCs retained.

    good_vertex_mask : ndarray of bool
        Identifies vertices included in PCA.

    Returns
    -------
    reconstructed_full : ndarray
        (epochs, original_vertices, times)

        Removed/bad vertices are restored as NaN so the original
        vertex dimension and ordering are preserved.

    roi_tc : ndarray
        Mean across valid, sign-aligned reconstructed vertices:
        (epochs, 1, times)
    """

    n_epochs, n_original_vertices, n_times = roi_data_full.shape

    data_good = roi_data_full[:, good_vertex_mask, :]

    X_full = (
        data_good
        .transpose(0, 2, 1)
        .reshape(
            n_epochs * n_times,
            data_good.shape[1],
        )
    )

    # Project full epochs through the PCA model fitted on 0–0.5 s.
    scores = pca.transform(X_full)

    # Keep only the selected leading PCs.
    scores_reduced = np.zeros_like(scores)
    scores_reduced[:, :n_keep] = scores[:, :n_keep]

    # Inverse-transform back into the retained vertex space.
    X_reconstructed_good = pca.inverse_transform(
        scores_reduced
    )

    reconstructed_good = (
        X_reconstructed_good
        .reshape(
            n_epochs,
            n_times,
            data_good.shape[1],
        )
        .transpose(0, 2, 1)
    )

    # Restore the ORIGINAL ROI vertex dimension.
    # Vertices excluded due to NaN/Inf remain NaN.
    reconstructed_full = np.full(
        (
            n_epochs,
            n_original_vertices,
            n_times,
        ),
        np.nan,
        dtype=reconstructed_good.dtype,
    )

    reconstructed_full[
        :,
        good_vertex_mask,
        :
    ] = reconstructed_good

    # ROI representative time course.
    # keepdims=True deliberately preserves a singleton "vertex" axis
    # required by the downstream ERPAC code.
    roi_tc = np.nanmean(
        reconstructed_full,
        axis=1,
        keepdims=True,
    )

    return reconstructed_full, roi_tc


def make_all_vtx_output_dir(
    group,
    subject,
    task,
    stage,
    block=None,
    adaptation_split=None,
):
    """
    Requested hierarchy:

    group -> sub -> task -> task_stage
          -> block (DeCRAT)
          -> adaptation stage (adaptation block only)
    """

    out_dir = (
        STCS_ALL_VTX_DIR
        / str(group)
        / str(subject)
        / str(task)
        / str(stage)
    )

    if block is not None:
        out_dir = out_dir / str(block)

    if adaptation_split is not None:
        out_dir = out_dir / str(adaptation_split)

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    return out_dir


# ============================================================
# MAIN PCA + RECONSTRUCTION FOR ONE SUBJECT × ROI
# ============================================================

def run_subject_roi_pca(
    subject,
    group,
    roi_name,
):

    print("\n")
    print("=" * 70)
    print(
        f"PCA: {subject} | "
        f"{group} | "
        f"{roi_name}"
    )
    print("=" * 70)

    roi_label = get_combined_roi_label(
        roi_name
    )

    # Full, sign-aligned epochs are kept separately because
    # different conditions can have different numbers of time points.
    condition_store = {}

    metadata_rows = []

    reference_vertices_lh = None
    reference_vertices_rh = None
    reference_vertices = None
    reference_flip = None
    reference_pca_times = None

    global_epoch_idx = 0


    # ========================================================
    # LOAD + STORE EVERY CONDITION
    # ========================================================

    for condition in CONDITIONS:

        task = condition["task"]
        stage = condition["task_stage"]
        block = condition["task_block"]
        adaptation_split = condition["adaptation_split"]

        condition_name = make_condition_name(
            task,
            stage,
            block,
            adaptation_split,
        )

        print(f"\nLoading {condition_name}")

        stcs = load_condition_stcs(
            subject=subject,
            task=task,
            task_stage=stage,
            task_block=block,
            adaptation_split=adaptation_split,
        )

        print(f"  epochs found: {len(stcs)}")

        if len(stcs) == 0:
            print("  -> skipped")
            continue

        # ----------------------------------------------------
        # Preserve original source-file epoch indices.
        # ----------------------------------------------------
        original_epoch_indices = np.arange(
            len(stcs),
            dtype=int,
        )

        # ----------------------------------------------------
        # Existing FTT balancing step.
        # ----------------------------------------------------
        if task == "FTT":

            n_ftt_before = len(stcs)

            stcs, selected_ftt_indices = subsample_ftt_stcs(
                stcs=stcs,
                subject=subject,
                task_stage=stage,
            )

            original_epoch_indices = selected_ftt_indices

            print(
                f"  FTT subsampling: "
                f"{n_ftt_before} -> {len(stcs)} epochs "
                f"({FTT_KEEP_FRACTION:.3f} retained)"
            )

        # ----------------------------------------------------
        # Extract sign-aligned ROI vertices.
        # ----------------------------------------------------
        (
            roi_data,
            vertices_lh,
            vertices_rh,
            flip,
        ) = extract_roi_epoch_data(
            stcs,
            roi_label,
            src,
        )

        vertices = np.concatenate(
            [
                vertices_lh,
                vertices_rh,
            ]
        )

        # ----------------------------------------------------
        # Verify spatial feature space is identical across
        # every condition for this subject × ROI.
        # ----------------------------------------------------
        if reference_vertices is None:

            reference_vertices_lh = vertices_lh.copy()
            reference_vertices_rh = vertices_rh.copy()
            reference_vertices = vertices.copy()
            reference_flip = flip.copy()

        else:

            if not np.array_equal(
                vertices,
                reference_vertices,
            ):
                raise RuntimeError(
                    f"Vertex mismatch between conditions for "
                    f"{subject} | {roi_name}"
                )

            if not np.array_equal(
                flip,
                reference_flip,
            ):
                raise RuntimeError(
                    f"Sign-flip mismatch between conditions for "
                    f"{subject} | {roi_name}"
                )

        # ----------------------------------------------------
        # PCA fitting window only.
        # Full roi_data remains untouched.
        # ----------------------------------------------------
        times = stcs[0].times.copy()

        time_mask = get_pca_window_mask(
            times
        )

        pca_times = times[time_mask]

        if reference_pca_times is None:
            reference_pca_times = pca_times.copy()

        else:
            if len(pca_times) != len(reference_pca_times):
                raise RuntimeError(
                    f"Different number of samples in the "
                    f"{PCA_TMIN:.1f}–{PCA_TMAX:.1f} s PCA window for "
                    f"{subject} | {condition_name}"
                )

            if not np.allclose(
                pca_times,
                reference_pca_times,
            ):
                raise RuntimeError(
                    f"PCA fitting time vectors differ for "
                    f"{subject} | {condition_name}"
                )

        print(
            f"  full epoch: "
            f"{times[0]:.4f} to {times[-1]:.4f} s | "
            f"{roi_data.shape[2]} samples"
        )

        print(
            f"  PCA fit: "
            f"{pca_times[0]:.4f} to {pca_times[-1]:.4f} s | "
            f"{len(pca_times)} samples"
        )

        condition_store[condition_name] = {
            "data": roi_data,
            "times": times,
            "time_mask_fit": time_mask,
            "task": task,
            "task_stage": stage,
            "task_block": block,
            "adaptation_split": adaptation_split,
            "original_epoch_indices": original_epoch_indices,
        }

        # ----------------------------------------------------
        # Recoverable epoch metadata.
        # ----------------------------------------------------
        for local_epoch_idx in range(
            roi_data.shape[0]
        ):

            metadata_rows.append(
                {
                    "subject": subject,
                    "group": group,
                    "roi": roi_name,
                    "task": task,
                    "task_stage": stage,
                    "task_block": (
                        block
                        if block is not None
                        else "NA"
                    ),
                    "adaptation_split": (
                        adaptation_split
                        if adaptation_split is not None
                        else "NA"
                    ),
                    "condition": condition_name,
                    "condition_epoch_index": local_epoch_idx,
                    "original_epoch_index": int(
                        original_epoch_indices[local_epoch_idx]
                    ),
                    "global_epoch_index": global_epoch_idx,
                    "tmin": float(times[0]),
                    "tmax": float(times[-1]),
                    "n_times": int(len(times)),
                }
            )

            global_epoch_idx += 1


    # ========================================================
    # CHECK DATA
    # ========================================================

    if len(condition_store) == 0:

        print(
            f"No usable data for "
            f"{subject} | {roi_name}"
        )

        return None


    # ========================================================
    # IDENTIFY INVALID VERTICES ACROSS ALL FULL CONDITIONS
    # ========================================================
    #
    # Conditions are NOT concatenated here because n_times can differ.
    #
    # A vertex is excluded from PCA if it contains ANY NaN/Inf in any
    # condition. It is restored as NaN in reconstructed_full so that
    # the requested original vertex dimension/order is preserved.
    # ========================================================

    n_original_vertices = len(
        reference_vertices
    )

    bad_vertex_mask = np.zeros(
        n_original_vertices,
        dtype=bool,
    )

    for item in condition_store.values():

        data_condition = item["data"]

        bad_vertex_mask |= np.any(
            ~np.isfinite(data_condition),
            axis=(0, 2),
        )

    good_vertex_mask = ~bad_vertex_mask

    n_bad_vertices = int(
        bad_vertex_mask.sum()
    )

    n_good_vertices = int(
        good_vertex_mask.sum()
    )

    print(
        f"\nOriginal ROI vertices : "
        f"{n_original_vertices}"
    )
    print(
        f"Bad vertices removed   : "
        f"{n_bad_vertices}"
    )
    print(
        f"Vertices used for PCA  : "
        f"{n_good_vertices}"
    )

    if n_good_vertices < 2:
        raise RuntimeError(
            f"Too few valid vertices for PCA: "
            f"{subject} | {roi_name}"
        )


    # ========================================================
    # BUILD POOLED PCA-FITTING MATRIX
    # ========================================================
    #
    # For every condition:
    #     full epoch -> 0–0.5 s only
    #     epochs × good_vertices × time
    #                  ↓
    #     (epochs × time) × good_vertices
    #
    # Then concatenate only these 2-D matrices.
    # ========================================================

    X_fit_conditions = []

    for condition_name, item in condition_store.items():

        data_fit = item["data"][
            :,
            good_vertex_mask,
            :
        ][:, :, item["time_mask_fit"]]

        n_epochs_condition = data_fit.shape[0]
        n_times_fit = data_fit.shape[2]

        X_condition = (
            data_fit
            .transpose(0, 2, 1)
            .reshape(
                n_epochs_condition * n_times_fit,
                n_good_vertices,
            )
        )

        X_fit_conditions.append(
            X_condition
        )

    X_fit = np.concatenate(
        X_fit_conditions,
        axis=0,
    )

    print(
        "PCA fitting matrix shape:",
        X_fit.shape,
    )


    # ========================================================
    # FIT FULL PCA DECOMPOSITION
    # ========================================================

    pca = PCA(
        n_components=None,
        whiten=False,
        svd_solver="full",
    )

    pca.fit(
        X_fit
    )

    explained = (
        pca.explained_variance_ratio_
    )

    cumulative = np.cumsum(
        explained
    )

    n_80 = components_for_variance(
        explained,
        0.80,
    )
    n_90 = components_for_variance(
        explained,
        0.90,
    )
    n_95 = components_for_variance(
        explained,
        0.95,
    )
    n_99 = components_for_variance(
        explained,
        0.99,
    )

    n_keep = components_for_variance(
        explained,
        PCA_VARIANCE_THRESHOLD,
    )

    actual_variance_kept = float(
        cumulative[n_keep - 1]
    )

    print("\nPCA RESULTS")
    print("-" * 50)
    print(
        f"PC1 variance: "
        f"{explained[0] * 100:.2f}%"
    )

    if len(explained) > 1:
        print(
            f"PC2 variance: "
            f"{explained[1] * 100:.2f}%"
        )

    if len(explained) > 2:
        print(
            f"PC3 variance: "
            f"{explained[2] * 100:.2f}%"
        )

    print(f"PCs for 80%: {n_80}")
    print(f"PCs for 90%: {n_90}")
    print(f"PCs for 95%: {n_95}")
    print(f"PCs for 99%: {n_99}")

    print(
        f"\nRetaining {n_keep} PCs "
        f"for target >= "
        f"{PCA_VARIANCE_THRESHOLD * 100:.1f}% "
        f"(actual = {actual_variance_kept * 100:.3f}%)"
    )


    # ========================================================
    # COMPONENT OUTPUTS: sub -> ROI
    # ========================================================

    subject_roi_dir = (
        COMPONENTS_DIR
        / str(subject)
        / roi_name
    )

    subject_roi_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    metadata_df = pd.DataFrame(
        metadata_rows
    )

    metadata_df.to_csv(
        subject_roi_dir
        / "epoch_metadata.csv",
        index=False,
    )

    variance_df = pd.DataFrame(
        {
            "component": np.arange(
                1,
                len(explained) + 1,
            ),
            "explained_variance_ratio":
                explained,
            "explained_variance_percent":
                explained * 100,
            "cumulative_variance_ratio":
                cumulative,
            "cumulative_variance_percent":
                cumulative * 100,
            "retained_primary": np.arange(
                1,
                len(explained) + 1,
            ) <= n_keep,
        }
    )

    variance_df.to_csv(
        subject_roi_dir
        / "explained_variance.csv",
        index=False,
    )

    # PCA loadings/components + all parameters required to reproduce
    # the transformation later.
    np.savez_compressed(
        subject_roi_dir
        / "pca_model_parameters.npz",

        components=pca.components_,
        mean=pca.mean_,
        explained_variance=
            pca.explained_variance_,
        explained_variance_ratio=
            pca.explained_variance_ratio_,
        singular_values=
            pca.singular_values_,

        variance_threshold=
            PCA_VARIANCE_THRESHOLD,
        n_components_retained=
            n_keep,
        actual_variance_retained=
            actual_variance_kept,

        vertices_lh=
            reference_vertices_lh,
        vertices_rh=
            reference_vertices_rh,
        vertices_concatenated=
            reference_vertices,

        good_vertex_mask=
            good_vertex_mask,
        bad_vertex_mask=
            bad_vertex_mask,
        sign_flip=
            reference_flip,

        pca_fit_times=
            reference_pca_times,
        pca_tmin=
            PCA_TMIN,
        pca_tmax=
            PCA_TMAX,
    )


    # ========================================================
    # RECONSTRUCT + SAVE EVERY CONDITION
    # ========================================================

    for condition_name, item in condition_store.items():

        reconstructed_full, roi_tc = (
            reconstruct_condition(
                roi_data_full=item["data"],
                pca=pca,
                n_keep=n_keep,
                good_vertex_mask=
                    good_vertex_mask,
            )
        )

        print(
            f"\n{condition_name}"
        )
        print(
            f"  reconstructed: "
            f"{reconstructed_full.shape}"
        )
        print(
            f"  ROI tc: "
            f"{roi_tc.shape}"
        )

        # ----------------------------------------------------
        # A. Reconstructed vertices:
        #
        # group -> sub -> task -> stage -> block -> split
        # ----------------------------------------------------

        all_vtx_dir = make_all_vtx_output_dir(
            group=group,
            subject=subject,
            task=item["task"],
            stage=item["task_stage"],
            block=item["task_block"],
            adaptation_split=
                item["adaptation_split"],
        )

        all_vtx_path = (
            all_vtx_dir
            / f"{condition_name}_{roi_name}_pca_reconstructed.npz"
        )

        np.savez_compressed(
            all_vtx_path,

            data=reconstructed_full,
            times=item["times"],

            vertices_lh=
                reference_vertices_lh,
            vertices_rh=
                reference_vertices_rh,
            vertices_concatenated=
                reference_vertices,

            good_vertex_mask=
                good_vertex_mask,
            bad_vertex_mask=
                bad_vertex_mask,

            subject=subject,
            group=group,
            roi=roi_name,
            task=item["task"],
            task_stage=
                item["task_stage"],
            task_block=(
                item["task_block"]
                if item["task_block"] is not None
                else "NA"
            ),
            adaptation_split=(
                item["adaptation_split"]
                if item["adaptation_split"] is not None
                else "NA"
            ),

            original_epoch_indices=
                item["original_epoch_indices"],

            variance_threshold=
                PCA_VARIANCE_THRESHOLD,
            n_components_retained=
                n_keep,
            actual_variance_retained=
                actual_variance_kept,
        )

        # ----------------------------------------------------
        # B. One ROI representative signal:
        #
        # group -> sub
        #
        # Filename preserves full condition + ROI information.
        # Shape remains:
        # epochs × 1 × times
        # ----------------------------------------------------

        ave_vtx_dir = (
            STCS_AVE_VTX_DIR
            / str(group)
            / str(subject)
        )

        ave_vtx_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        roi_tc_path = (
            ave_vtx_dir
            / f"{condition_name}_{roi_name}_pca_roi_tc.npz"
        )

        np.savez_compressed(
            roi_tc_path,

            data=roi_tc,
            times=item["times"],

            subject=subject,
            group=group,
            roi=roi_name,
            task=item["task"],
            task_stage=
                item["task_stage"],
            task_block=(
                item["task_block"]
                if item["task_block"] is not None
                else "NA"
            ),
            adaptation_split=(
                item["adaptation_split"]
                if item["adaptation_split"] is not None
                else "NA"
            ),

            original_epoch_indices=
                item["original_epoch_indices"],

            variance_threshold=
                PCA_VARIANCE_THRESHOLD,
            n_components_retained=
                n_keep,
            actual_variance_retained=
                actual_variance_kept,
        )


    # ========================================================
    # HUMAN-READABLE PCA LOG
    # ========================================================

    log_path = (
        subject_roi_dir
        / "pca_summary.txt"
    )

    with open(
        log_path,
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            "SOURCE-LEVEL PCA DENOISING SUMMARY\n"
        )
        f.write(
            "=" * 60 + "\n\n"
        )

        f.write(
            f"Subject: {subject}\n"
        )
        f.write(
            f"Group: {group}\n"
        )
        f.write(
            f"ROI: {roi_name}\n\n"
        )

        f.write(
            f"PCA fitting window: "
            f"{PCA_TMIN:.3f}–{PCA_TMAX:.3f} s\n"
        )
        f.write(
            f"PCA time points per epoch: "
            f"{len(reference_pca_times)}\n"
        )
        f.write(
            f"PCA observations: "
            f"{X_fit.shape[0]}\n"
        )

        f.write(
            f"Original ROI vertices: "
            f"{n_original_vertices}\n"
        )
        f.write(
            f"Bad vertices excluded from PCA: "
            f"{n_bad_vertices}\n"
        )
        f.write(
            f"Vertices used for PCA: "
            f"{n_good_vertices}\n\n"
        )

        f.write(
            "DENOISING RULE\n"
        )
        f.write(
            "-" * 60 + "\n"
        )
        f.write(
            f"Target retained variance: "
            f"{PCA_VARIANCE_THRESHOLD * 100:.1f}%\n"
        )
        f.write(
            f"PCs retained: "
            f"{n_keep}\n"
        )
        f.write(
            f"Actual retained variance: "
            f"{actual_variance_kept * 100:.3f}%\n"
        )
        f.write(
            f"Discarded variance: "
            f"{(1 - actual_variance_kept) * 100:.3f}%\n\n"
        )

        f.write(
            "CONDITION COUNTS\n"
        )
        f.write(
            "-" * 60 + "\n"
        )

        condition_counts = (
            metadata_df
            .groupby(
                "condition",
                dropna=False,
            )
            .size()
            .rename("n_epochs")
        )

        f.write(
            condition_counts.to_string()
        )
        f.write("\n\n")

        f.write(
            "EXPLAINED VARIANCE\n"
        )
        f.write(
            "-" * 60 + "\n"
        )

        for i, var in enumerate(
            explained,
            start=1,
        ):
            retained_flag = (
                " *retained"
                if i <= n_keep
                else ""
            )

            f.write(
                f"PC{i:02d}: "
                f"{var * 100:8.3f}%   "
                f"cumulative = "
                f"{cumulative[i - 1] * 100:8.3f}%"
                f"{retained_flag}\n"
            )

        f.write("\n")
        f.write(
            f"PCs required for 80%: "
            f"{n_80}\n"
        )
        f.write(
            f"PCs required for 90%: "
            f"{n_90}\n"
        )
        f.write(
            f"PCs required for 95%: "
            f"{n_95}\n"
        )
        f.write(
            f"PCs required for 99%: "
            f"{n_99}\n"
        )


    print(
        f"\nSaved PCA components/diagnostics to:\n"
        f"{subject_roi_dir}"
    )

    return {
        "subject": subject,
        "group": group,
        "roi": roi_name,
        "n_original_vertices":
            n_original_vertices,
        "n_good_vertices":
            n_good_vertices,
        "n_bad_vertices":
            n_bad_vertices,
        "pc1_variance":
            explained[0],
        "pc2_variance": (
            explained[1]
            if len(explained) > 1
            else np.nan
        ),
        "pc3_variance": (
            explained[2]
            if len(explained) > 2
            else np.nan
        ),
        "n_components_80":
            n_80,
        "n_components_90":
            n_90,
        "n_components_95":
            n_95,
        "n_components_99":
            n_99,
        "variance_threshold":
            PCA_VARIANCE_THRESHOLD,
        "n_components_retained":
            n_keep,
        "actual_variance_retained":
            actual_variance_kept,
    }


# ============================================================
# RUN ALL SUBJECTS × ROIS
# ============================================================

summary_results = []

for group in GROUPS:

    print("\n")
    print("#" * 70)
    print(
        f"GROUP: {group}"
    )
    print("#" * 70)

    for subject in SUBJECTS[group]:

        for roi_name in ROI.keys():

            result = run_subject_roi_pca(
                subject=subject,
                group=group,
                roi_name=roi_name,
            )

            if result is not None:
                summary_results.append(
                    result
                )


# ============================================================
# SAVE ALL-SUBJECT PCA SUMMARY
# ============================================================

pca_summary_df = pd.DataFrame(
    summary_results
)

summary_path = (
    COMPONENTS_DIR
    / "pca_summary_all_subjects.csv"
)

pca_summary_df.to_csv(
    summary_path,
    index=False,
)

print("\n")
print("=" * 70)
print(
    "PCA DENOISING + RECONSTRUCTION COMPLETE"
)
print("=" * 70)

print(
    f"\nSummary saved to:\n"
    f"{summary_path}"
)

print(
    f"\nOutput root:\n"
    f"{PCA_ROOT}"
)

______________________
Dirty field: epochs num checks

In [ ]:
adaptation_early = []
epochs_num = []
for group, sub, task, stage, block, adaptation_split in iterate_dataset(
        GROUPS,
        SUBJECTS,
        TASKS,
        TASK_STAGES,
        TASK_BLOCKS,
        ADAPTATION_SPLITS
    ):
    # Load epochs data for the current subject, group, task stage, and task block
    if block is not None and adaptation_split is not None:
        epochs_folder = os.path.join(SOURCE_DATA_DIR, group, sub, task, stage, block, adaptation_split)
    elif block is not None and adaptation_split is None:
        epochs_folder = os.path.join(SOURCE_DATA_DIR, group, sub, task, stage, block)
    else:
        epochs_folder = os.path.join(SOURCE_DATA_DIR, group, sub, task, stage)
    print(f"{group}_{sub}_{task}_{stage}_{block}_{adaptation_split}: {len(os.listdir(epochs_folder))/2}")
    epochs_num.append(len(os.listdir(epochs_folder))/2)
    if adaptation_split == 'early':
        adaptation_early.append(len(os.listdir(epochs_folder))/2)
print("EPOCHS COUNTS:", pd.Series(epochs_num).describe(), sep="\n")

Y_s1_pac_sub01_FTT_plan_None_None: 99.0
Y_s1_pac_sub01_FTT_go_None_None: 84.0
Y_s1_pac_sub01_DeCRAT_plan_baseline_None: 50.0
Y_s1_pac_sub01_DeCRAT_plan_adaptation_early: 45.0
Y_s1_pac_sub01_DeCRAT_plan_adaptation_late: 45.0
Y_s1_pac_sub01_DeCRAT_go_baseline_None: 62.0
Y_s1_pac_sub01_DeCRAT_go_adaptation_early: 61.0
Y_s1_pac_sub01_DeCRAT_go_adaptation_late: 61.0
Y_s1_pac_sub07_FTT_plan_None_None: 102.0
Y_s1_pac_sub07_FTT_go_None_None: 98.0
Y_s1_pac_sub07_DeCRAT_plan_baseline_None: 47.0
Y_s1_pac_sub07_DeCRAT_plan_adaptation_early: 45.0
Y_s1_pac_sub07_DeCRAT_plan_adaptation_late: 45.0
Y_s1_pac_sub07_DeCRAT_go_baseline_None: 57.0
Y_s1_pac_sub07_DeCRAT_go_adaptation_early: 61.0
Y_s1_pac_sub07_DeCRAT_go_adaptation_late: 61.0
Y_s1_pac_sub10_FTT_plan_None_None: 108.0
Y_s1_pac_sub10_FTT_go_None_None: 106.0
Y_s1_pac_sub10_DeCRAT_plan_baseline_None: 48.0
Y_s1_pac_sub10_DeCRAT_plan_adaptation_early: 46.0
Y_s1_pac_sub10_DeCRAT_plan_adaptation_late: 46.0
Y_s1_pac_sub10_DeCRAT_go_baseline_None: 57.0


In [5]:
119/100*70

83.3

In [10]:
print(min(adaptation_early))
print(max(adaptation_early))

37.0
63.0
